# 06 · Overfitting, dropout and weight decay

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/06_overfitting_dropout.ipynb) &nbsp; 📖 Concept deck: [Regularization and generalization](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/08-regularization/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** overfitting with polynomial regression, inverted dropout with the right scaling, and ridge (weight decay) as a fix, measured on held-out data.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


r0 = np.random.default_rng(0)
def make_data(n, seed):
    g = np.random.default_rng(seed); x = g.uniform(-1, 1, n)
    return x, np.sin(3 * x) + 0.2 * g.normal(size=n)
xtr, ytr = make_data(15, 1); xte, yte = make_data(500, 2)
def features(x, degree): return np.vander(x, degree + 1, increasing=True)

## Task 1: Fit polynomials of growing degree

Fit a least-squares polynomial of `degree` to the 15 training points (use `np.linalg.lstsq` on `features`), with optional ridge penalty `lam` (minimise $\lVert \Phi w - y\rVert^2 + \lambda \lVert w \rVert^2$).
Return `(train_mse, test_mse)`.

In [ ]:
def fit_poly(degree, lam=0.0):
    # TODO (ridge: solve (ΦᵀΦ + λI) w = Φᵀy)
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_fit_poly(f):
    tr3, te3 = f(3); tr14, te14 = f(14)
    assert tr14 < 1e-6, "degree 14 interpolates 15 points: train error ≈ 0"
    assert te14 > 10 * te3, "…but its test error explodes: overfitting"
    trr, ter = f(14, lam=1e-3)
    assert ter < te14 / 10, "a small ridge penalty tames the degree-14 fit"

check("fit_poly", _test_fit_poly, fit_poly)

## Task 2: Inverted dropout

In training mode, zero each element of `h` with probability `p` and **divide the survivors by (1 − p)** so the expected value is unchanged; in eval mode, return `h` untouched. Return `(out, mask)` (mask is all ones in eval).

In [ ]:
def dropout(h, p, train, rng):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_dropout(f):
    g = np.random.default_rng(0); h = np.ones((2000, 100))
    out, m = f(h, 0.5, True, g)
    assert abs((m == 0).mean() - 0.5) < 0.02, "about half the units dropped"
    assert abs(out.mean() - 1) < 0.02, "inverted scaling keeps the mean at 1"
    e, _ = f(h, 0.5, False, g); assert np.array_equal(e, h), "eval mode is the identity"

check("dropout", _test_dropout, dropout)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Fit polynomials of growing degree
def fit_poly_solution(degree, lam=0.0):
    P = features(xtr, degree)
    w = np.linalg.lstsq(P, ytr, rcond=None)[0] if lam == 0 else np.linalg.solve(P.T @ P + lam * np.eye(degree + 1), P.T @ ytr)
    return np.mean((P @ w - ytr) ** 2), np.mean((features(xte, degree) @ w - yte) ** 2)

check("fit_poly (reference)", _test_fit_poly, fit_poly_solution)

In [ ]:
# @title Solution for Task 2: Inverted dropout
def dropout_solution(h, p, train, rng):
    if not train:
        return h, np.ones_like(h)
    m = (rng.random(h.shape) >= p).astype(h.dtype)
    return h * m / (1 - p), m

check("dropout (reference)", _test_dropout, dropout_solution)

---
## Experiments
**The U-shaped curve.** Train and test error as the polynomial degree grows, with and without a small ridge penalty.

In [ ]:
for d in [1, 3, 5, 9, 12, 14]:
    tr, te = fit_poly_solution(d); trr, ter = fit_poly_solution(d, 1e-3)
    print(f"degree {d:2d}: train {tr:.4f} test {te:10.4f}   |  ridge 1e-3: train {trr:.4f} test {ter:.4f}")

---
## Takeaways
* More capacity always lowers training error; test error falls then rises when the model fits noise.
* Weight decay (ridge) and dropout limit how much the model can bend to individual points.
* Inverted dropout scales by 1/(1 − p) during training so nothing changes at inference.